In [1]:
"""
Prepare restored peatland training points and combine them with active/abandoned training data.

This script:
- samples raster predictor values at restored-area points,
- assigns restored points to class 2,
- combines restored, active and abandoned point samples,
- removes overlapping points with priority: restored > abandoned > active.

Classes:
0 = active peat extraction area
1 = abandoned peat extraction area
2 = restored peat extraction area
"""

import geopandas as gpd
import rasterio
from rasterio.transform import rowcol
from scipy.ndimage import uniform_filter
import pandas as pd
import numpy as np
from pathlib import Path

BASE_DIR = Path("path\to\work\folder")

restored_points_path = BASE_DIR / "restored_points_10_per_polygon.shp"
tiles_dir = BASE_DIR / "tiles"

out_restored = BASE_DIR / "restored_points_with_predictors.gpkg"

base_features = [
    "ndvi", "ndwi", "ndmi", "swir1", "swir2",
    "dem", "vv", "vh", "twi"
]

focal_features = ["dem", "vv", "vh", "twi"]
focal_sizes = [3, 5]

# Read restored points
pts = gpd.read_file(restored_points_path)
pts = pts.to_crs("EPSG:3301")

pts["label"] = 2
pts["class_name"] = "restored"


def sample_feature(points_gdf, feature_name):
    folder = tiles_dir / feature_name
    values = np.full(len(points_gdf), np.nan, dtype="float32")

    raster_paths = sorted(folder.glob("*.tif"))

    for raster_path in raster_paths:
        with rasterio.open(raster_path) as src:
            bounds = src.bounds

            mask = (
                (points_gdf.geometry.x >= bounds.left) &
                (points_gdf.geometry.x <= bounds.right) &
                (points_gdf.geometry.y >= bounds.bottom) &
                (points_gdf.geometry.y <= bounds.top)
            )

            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue

            coords = [
                (points_gdf.geometry.iloc[i].x, points_gdf.geometry.iloc[i].y)
                for i in idx
            ]

            sampled = list(src.sample(coords))
            vals = np.array([s[0] for s in sampled], dtype="float32")

            if src.nodata is not None:
                vals[vals == src.nodata] = np.nan

            values[idx] = vals

    return values

def focal_mean_std_ignore_nan(arr: np.ndarray, size: int = 3):
    valid = np.isfinite(arr).astype("float32")
    arr_zero = np.where(np.isfinite(arr), arr, 0).astype("float32")

    kernel_n = float(size * size)

    sum_arr = uniform_filter(arr_zero, size=size, mode="constant", cval=0.0) * kernel_n
    count_arr = uniform_filter(valid, size=size, mode="constant", cval=0.0) * kernel_n

    mean_arr = np.full(arr.shape, np.nan, dtype="float32")
    valid_mask = count_arr > 0
    mean_arr[valid_mask] = sum_arr[valid_mask] / count_arr[valid_mask]

    sumsq_arr = uniform_filter(arr_zero ** 2, size=size, mode="constant", cval=0.0) * kernel_n

    var_arr = np.full(arr.shape, np.nan, dtype="float32")
    var_arr[valid_mask] = (sumsq_arr[valid_mask] / count_arr[valid_mask]) - (mean_arr[valid_mask] ** 2)
    var_arr[var_arr < 0] = 0

    std_arr = np.full(arr.shape, np.nan, dtype="float32")
    std_arr[valid_mask] = np.sqrt(var_arr[valid_mask])

    return mean_arr, std_arr


def sample_focal_std(points_gdf, feature_name, size):
    folder = tiles_dir / feature_name
    values = np.full(len(points_gdf), np.nan, dtype="float32")

    raster_paths = sorted(folder.glob("*.tif"))

    for raster_path in raster_paths:
        with rasterio.open(raster_path) as src:
            bounds = src.bounds

            mask = (
                (points_gdf.geometry.x >= bounds.left) &
                (points_gdf.geometry.x <= bounds.right) &
                (points_gdf.geometry.y >= bounds.bottom) &
                (points_gdf.geometry.y <= bounds.top)
            )

            idx = np.where(mask)[0]
            if len(idx) == 0:
                continue

            arr = src.read(1).astype("float32")

            if src.nodata is not None:
                arr[arr == src.nodata] = np.nan

            _, std_arr = focal_mean_std_ignore_nan(arr, size=size)

            coords = [
                (points_gdf.geometry.iloc[i].x, points_gdf.geometry.iloc[i].y)
                for i in idx
            ]

            rows, cols = rowcol(src.transform, [c[0] for c in coords], [c[1] for c in coords])
            rows = np.asarray(rows)
            cols = np.asarray(cols)

            valid_idx = (
                (rows >= 0) & (rows < src.height) &
                (cols >= 0) & (cols < src.width)
            )

            idx_valid = idx[valid_idx]
            rows_valid = rows[valid_idx]
            cols_valid = cols[valid_idx]

            values[idx_valid] = std_arr[rows_valid, cols_valid]

    return values

# Sample base features
for f in base_features:
    print("Sampling", f)
    pts[f] = sample_feature(pts, f)

for size in focal_sizes:
    for f in focal_features:
        col_name = f"{f}_sd{size}"
        print("Sampling", col_name)
        pts[col_name] = sample_focal_std(pts, f, size)

focal_cols = [
    "dem_sd3", "vv_sd3", "vh_sd3", "twi_sd3",
    "dem_sd5", "vv_sd5", "vh_sd5", "twi_sd5"
]

all_predictors = base_features + focal_cols

# Drop rows where any predictor is missing
rows_before = len(pts)
pts = pts.dropna(subset=all_predictors).copy()
rows_after = len(pts)

print("Rows before dropna:", rows_before)
print("Rows after dropna:", rows_after)
print("Dropped rows:", rows_before - rows_after)

# Remove problematic ID columns
drop_cols = ["fid", "FID", "Id", "id"]
for col in drop_cols:
    if col in pts.columns:
        pts = pts.drop(columns=[col])

# Save restored points with predictors
pts.to_file(out_restored, layer="restored_points", driver="GPKG")

print("Saved restored:", out_restored)
print("Rows:", len(pts))
print(pts[base_features + ["label"]].head())


# -------------------------
# Combine with existing training points
# -------------------------
main_path = BASE_DIR / "training_points_with_predictors_peatmasked_focal.gpkg"
restored_path = out_restored

main = gpd.read_file(
    main_path,
    layer="points_with_predictors" # change layer name as needed
)

restored = gpd.read_file(
    restored_path,
    layer="restored_points"
)

main.columns = main.columns.str.lower()
restored.columns = restored.columns.str.lower()

restored = restored.to_crs(main.crs)
restored["label"] = 2

needed_cols = [
    "label",
    "ndvi", "ndwi", "ndmi", "swir1", "swir2",
    "dem", "vv", "vh", "twi",
    "dem_sd3", "vv_sd3", "vh_sd3", "twi_sd3",
    "dem_sd5", "vv_sd5", "vh_sd5", "twi_sd5",
    "geometry"
]

main = main[needed_cols]
restored = restored[needed_cols]

combined = pd.concat([main, restored], ignore_index=True)
combined = gpd.GeoDataFrame(combined, geometry="geometry", crs=main.crs)

# Remove nulls again after combining
combined = combined.dropna(subset=all_predictors + ["label"]).copy()

# -------------------------
# Remove overlapping points
# Priority: restored (2) > abandoned (1) > active (0)
# -------------------------

combined = combined.to_crs("EPSG:3301")

restored = combined[combined["label"] == 2].copy()
abandoned = combined[combined["label"] == 1].copy()
active = combined[combined["label"] == 0].copy()

# Points within 10 m of restored points are removed from active/abandoned classes
# to reduce conflicting labels near the same location.
overlap_buffer_m = 10

restored_buf = restored.copy()
restored_buf["geometry"] = restored_buf.geometry.buffer(overlap_buffer_m)

active_overlap = gpd.sjoin(
    active,
    restored_buf[["geometry"]],
    how="inner",
    predicate="within"
).index.unique()

abandoned_overlap = gpd.sjoin(
    abandoned,
    restored_buf[["geometry"]],
    how="inner",
    predicate="within"
).index.unique()

active_clean = active.drop(index=active_overlap)
abandoned_clean = abandoned.drop(index=abandoned_overlap)

combined_clean = pd.concat(
    [active_clean, abandoned_clean, restored],
    ignore_index=True
)

combined_clean = gpd.GeoDataFrame(
    combined_clean,
    geometry="geometry",
    crs=combined.crs
)

print("Before overlap removal:")
print(combined["label"].value_counts())

print("After overlap removal:")
print(combined_clean["label"].value_counts())

drop_cols = ["fid", "FID", "Id", "id", "index_right"]

for col in drop_cols:
    if col in combined_clean.columns:
        combined_clean = combined_clean.drop(columns=[col])

out_combined = BASE_DIR / "training_points_with_predictors_all_classes.gpkg"

if out_combined.exists():
    out_combined.unlink()

combined_clean.to_file(out_combined, layer="points", driver="GPKG")

print("Saved combined:", out_combined)
print(combined_clean["label"].value_counts())

Sampling ndvi
Sampling ndwi
Sampling ndmi
Sampling swir1
Sampling swir2
Sampling dem
Sampling vv
Sampling vh
Sampling twi
Sampling dem_sd3
Sampling vv_sd3
Sampling vh_sd3
Sampling twi_sd3
Sampling dem_sd5
Sampling vv_sd5
Sampling vh_sd5
Sampling twi_sd5
Rows before dropna: 456
Rows after dropna: 307
Dropped rows: 149
Saved restored: C:\Users\Egert\Documents\peatland_detection\restored_points_with_predictors.gpkg
Rows: 307
       ndvi      ndwi      ndmi   swir1  swir2        dem         vv  \
0  0.701701 -0.651291  0.231986  1311.0  676.0  67.752167  -8.673466   
1  0.709587 -0.642023  0.274522  1326.0  694.0  67.551422  -8.818522   
2  0.742381 -0.627942  0.250591  1268.0  621.0  67.367508 -10.140621   
3  0.792052 -0.671624  0.358846   959.5  489.0  87.262825  -8.827880   
4  0.777544 -0.663856  0.316375  1009.5  503.5  88.265686  -8.999008   

          vh       twi  label  
0 -15.429452  7.543183      2  
1 -16.025936  6.356637      2  
2 -16.012863  6.740722      2  
3 -14.830061 